In [ ]:
import numpy as np
import pandas as pd

In [ ]:
FILES = [
"I1.csv",
"I2.csv",
"I3.csv",
"I4.csv",
"I5.csv",
"I6.csv",
"I7.csv",
"I8.csv",
"I9.csv",
"I10.csv",
"I11.csv",
"I12.csv",
"I13.csv",
"I14.csv",
"I15.csv",
"I16.csv",
"I17.csv",
"I18.csv",
"I19.csv",
"I20.csv",
"I21.csv",
"I22.csv",
"I23.csv",
"I24.csv",
]
frames = []
for fp in FILES:
    df_i = pd.read_csv(fp, low_memory=False, on_bad_lines="warn")
    frames.append(df_i)

df_raw = pd.concat(frames, ignore_index=True)

df_raw.sample(n=20)

In [ ]:
need = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "MW", "B365H", "B365D", "B365A"]
have = [c for c in need if c in df_raw.columns]
df = df_raw[have].copy()
df.sample(n=10)

In [ ]:
df["Date"] = df["Date"].astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

df["date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

m = df["date"].isna() & df["Date"].notna()

if m.any():
    # second pass for leftovers (rare alt formats)
    df.loc[m, "date"] = pd.to_datetime(df.loc[m, "Date"], errors="coerce", dayfirst=False)

# --- Season = START YEAR ONLY (e.g., 2001 for 2001/02) ---
season = df["date"].dt.year
season = season.where(df["date"].dt.month >= 8, season - 1)
df["season"] = season.astype("Int64")

In [ ]:
df = df.rename(columns={"HomeTeam": "home_team", "AwayTeam": "away_team"})

df["hometeamgoals"] = pd.to_numeric(df.get("FTHG"), errors="coerce").astype("Int64")
df["awayteamgoals"] = pd.to_numeric(df.get("FTAG"), errors="coerce").astype("Int64")

win_mask   = df["hometeamgoals"].gt(df["awayteamgoals"]).fillna(False)
loss_mask  = df["hometeamgoals"].lt(df["awayteamgoals"]).fillna(False)
known_mask = df["hometeamgoals"].notna() & df["awayteamgoals"].notna()

df["hometeamresult"] = pd.NA
df.loc[win_mask,  "hometeamresult"] = "win"
df.loc[loss_mask, "hometeamresult"] = "loss"
df.loc[~win_mask & ~loss_mask & known_mask, "hometeamresult"] = "draw"

df["home_team_points"] = pd.NA
df["away_team_points"] = pd.NA
df.loc[win_mask,  ["home_team_points", "away_team_points"]] = [3, 0]
df.loc[loss_mask, ["home_team_points", "away_team_points"]] = [0, 3]
df.loc[~win_mask & ~loss_mask & known_mask, ["home_team_points", "away_team_points"]] = [1, 1]

df[["home_team_points","away_team_points"]] = df[["home_team_points","away_team_points"]].astype("Int64")


In [ ]:
if "MW" in df.columns:
    df = df.rename(columns={"MW": "week"})

odds_map = {"B365H": "OddHome", "B365D": "OddDraw", "B365A": "OddAway"}
for src, dst in odds_map.items():
    if src in df.columns:
        df = df.rename(columns={src: dst})
        df[dst] = pd.to_numeric(df[dst], errors="coerce")

final_cols = ["season", "date"]
if "week" in df.columns:
    final_cols.append("week")


final_cols += [
    "home_team", "away_team",
    "hometeamgoals", "awayteamgoals",
    "hometeamresult", "home_team_points", "away_team_points",
    "OddHome", "OddDraw", "OddAway",
]

final_df = df[[c for c in final_cols if c in df.columns]] \
    .sort_values(["season", "date"], ascending=[False, False], ignore_index=True)

In [ ]:
final_df.to_csv("final_matches.csv", index=False)

In [1]:
import pandas as pd
import numpy as np

def get_result(result):
    if pd.isna(result):
        return None
    if result == "win":  # 
        return 1 #
    if result == "draw":  
        return 0 #/
    else:  
        return -1 #

final_df = pd.read_csv("/Users/joeyli/skillvsluck/data/european_soccer/data/processed/serie_a.csv")
final_df["hometeamresult"] = final_df["hometeamresult"].apply(get_result)
final_df.to_csv("serie_a_resultsasint.csv", index = False)